# Video Game Sales & Engagement Analysis - PHASE 2

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import pyodbc
import urllib



IMPORT THE CLEAN DATASET Engagement

In [2]:
df_CleanEngagementData = pd.read_csv('CleanEngagementData.csv')
df_CleanEngagementData.head(2)

,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Title_norm
0,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring
1,Hades,2019-12-10,['Supergiant Games'],4.3,2900.0,2900.0,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,hades


# Check for Duplicates and fix

In [7]:
df_CleanEngagementData[df_CleanEngagementData.duplicated()]

,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Title_norm
326,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring
327,Hades,2019-12-10,['Supergiant Games'],4.3,2900.0,2900.0,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,hades
328,The Legend of Zelda: Breath of the Wild,2017-03-03,"['Nintendo', 'Nintendo EPD Production Group No...",4.4,4300.0,4300.0,"['Adventure', 'RPG']",The Legend of Zelda: Breath of the Wild is the...,['This game is the game (that is not CS:GO) th...,30000.0,2500.0,5000.0,2600.0,the legend of zelda breath of the wild
329,Undertale,2015-09-15,"['tobyfox', '8-4']",4.2,3500.0,3500.0,"['Adventure', 'Indie', 'RPG', 'Turn Based Stra...","A small child falls into the Underground, wher...",['soundtrack is tied for #1 with nier automata...,28000.0,679.0,4900.0,1800.0,undertale
330,Hollow Knight,2017-02-24,['Team Cherry'],4.4,3000.0,3000.0,"['Adventure', 'Indie', 'Platform']",A 2D metroidvania with an emphasis on close co...,"[""this games worldbuilding is incredible, with...",21000.0,2400.0,8300.0,2300.0,hollow knight
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1268,Bloodstained: Curse of the Moon,2018-05-23,['Inti Creates'],3.6,341.0,341.0,"['Adventure', 'Indie', 'Platform']",“Bloodstained: Curse of the Moon” is packed wi...,['Bloodstained me fez perceber que o que torna...,2300.0,41.0,800.0,397.0,bloodstained curse of the moon
1269,Final Fantasy XIII-2,2011-12-15,['Square Enix'],3.3,482.0,482.0,"['Adventure', 'RPG']",FINAL FANTASY XIII-2 is created with the aim o...,"[""Oh boy. Playing the XIII series is looking m...",2300.0,58.0,1400.0,449.0,final fantasy xiii2
1270,Agar.io,2015-04-28,"['Miniclip.com', 'Matheus Valadares']",2.2,81.0,81.0,"['Indie', 'Strategy']",Agar.io is a Massively-multiplayer top-down st...,"['""A Ganância que te move... É a mesma que te ...",4400.0,8.0,40.0,12.0,agario
1271,Fatal Frame II: Crimson Butterfly,2003-11-27,"['Tecmo Co., Ltd.', 'Ubisoft Entertainment']",4.2,398.0,398.0,['Adventure'],Crimson Butterfly is the second installment in...,['Pretty cool albeit a bit similar to the firs...,1000.0,38.0,690.0,513.0,fatal frame ii crimson butterfly


In [8]:
df_CleanEngagementData = df_CleanEngagementData.drop_duplicates()

In [11]:
df_CleanEngagementData.duplicated().sum()

np.int64(0)

In [12]:
df_CleanEngagementData[df_CleanEngagementData.duplicated(subset=['Title_norm'],keep=False)]


,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Title_norm
0,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring
5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2300.0,2300.0,"['Adventure', 'Simulator']",Minecraft focuses on allowing the player to ex...,['Minecraft is what you make of it. Unfortunat...,33000.0,1800.0,1100.0,230.0,minecraft
12,God of War,2018-04-20,"['Sony Interactive Entertainment', 'SIE Santa ...",4.2,2900.0,2900.0,"['Adventure', 'Brawler', 'RPG']",God of War is the sequel to God of War III as ...,"['freya te vejo como figura materna', 'i ruv !...",21000.0,1100.0,4800.0,2600.0,god of war
16,Yakuza 0,2015-03-12,"['Ryū Ga Gotoku Studios', 'Sega']",4.4,2700.0,2700.0,"['Adventure', 'Brawler', 'RPG', 'Simulator']","The glitz, glamour, and unbridled decadence of...",['THIS IS PEAK IT IS ONE OF MY FAVORITE GAMES ...,15000.0,1800.0,6400.0,2000.0,yakuza 0
21,Hi-Fi Rush,2023-01-25,"['Tango Gameworks', 'Bethesda Softworks']",4.3,926.0,926.0,"['Adventure', 'Brawler', 'Music', 'Platform']","As wannabe rockstar Chai, you’ll fight back ag...",['It was a great game all round ending felt a ...,3000.0,866.0,1500.0,2000.0,hifi rush
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1203,Tetris,1989-05-14,['Nintendo'],4.0,197.0,197.0,['Puzzle'],Tetris is a puzzle video game for the Game Boy...,['Bundling this gem with the OG Game Boy was t...,2500.0,18.0,76.0,21.0,tetris
1213,Tomb Raider,1996-10-25,"['Eidos Interactive', 'Core Design']",3.3,395.0,395.0,"['Adventure', 'Platform', 'Puzzle', 'Shooter']",Tomb Raider is a 3D action game developed by C...,"['kamera kontrol cringe', 'An ok game. I Platf...",2700.0,45.0,773.0,263.0,tomb raider
1282,Super Mario Sunshine,2020-09-18,"['Nintendo EAD', 'Nintendo']",3.7,19.0,19.0,"['Adventure', 'Platform']",A port of Super Mario Sunshine included in Sup...,['What an amazing remaster of an already amazi...,340.0,6.0,83.0,14.0,super mario sunshine
1332,Doom,2017-11-10,"['Bethesda Softworks', 'id Software']",3.9,80.0,80.0,['Shooter'],"Doom, the brutally fun and challenging modern-...",['can we get a Doom anime. Now I would watch t...,2300.0,37.0,393.0,150.0,doom


FactGameEngagement grain = ONE row per Game. For same Title_norm, we cannot keep multiple rows. <br>

# Create the Dataset to Star Schema (The Gold Standard) : PascalCase 

In [15]:
# Creating Reusable function for PascalCase conversion

import re 

def to_pascal_case(column_name):  
    column_name = column_name.replace('_', ' ') # Replace underscores with spaces
    column_name = re.sub(r'[^\w\s]', '', column_name)    # Remove special characters except spaces
    words = column_name.split()

    pascal_words = []
    
    for word in words:
        if word.isupper():     # If fully uppercase (like NA, EU, JP)
            pascal_words.append(word.title())
        else:
            pascal_words.append(word[0].upper() + word[1:])
    
    return ''.join(pascal_words)



In [16]:
# Apply to Entire DataFrame
df_CleanEngagementData.columns = [to_pascal_case(col) for col in df_CleanEngagementData.columns]

In [17]:
df_CleanEngagementData.head(2)

,Title,ReleaseDate,Team,Rating,TimesListed,NumberOfReviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,TitleNorm
0,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring
1,Hades,2019-12-10,['Supergiant Games'],4.3,2900.0,2900.0,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,hades


Different rows have: Different release dates Different metrics. If we keep only first: ❌ We lose engagement data ❌ We bias totals So we aggregate.

In [20]:
engagement_aggregated = (
    df_CleanEngagementData
    .groupby('TitleNorm', as_index=False)
    .agg({
        'Title': 'first',               # keep readable name
        'ReleaseDate': 'max',           # latest release
        'Rating': 'mean',               # average rating
        'TimesListed': 'sum',
        'NumberOfReviews': 'sum',
        'Plays': 'sum',
        'Playing': 'sum',
        'Backlogs': 'sum',
        'Wishlist': 'sum'
    })
)
engagement_aggregated['TitleNorm'].duplicated().sum()

np.int64(0)

In [21]:
print(len(df_CleanEngagementData))
print(len(engagement_aggregated))

1130
1098


# IMPORT THE CLEAN DATASET - Sales

In [22]:
df_CleanSalesData = pd.read_csv('CleanSalesData.csv')
df_CleanSalesData.head(2)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Name_norm
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros


In [23]:
df_CleanSalesData.columns = [to_pascal_case(col) for col in df_CleanSalesData.columns]
df_CleanSalesData.head(2)

,Rank,Name,Platform,Year,Genre,Publisher,NaSales,EuSales,JpSales,OtherSales,GlobalSales,NameNorm
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros


# Create DimGame (Master Game Table)

In [24]:
DimGame = pd.DataFrame(
    {
        'GameName': list(
            set(engagement_aggregated['TitleNorm'])
            .union(set(df_CleanSalesData['NameNorm']))
        )
    }
)


In [25]:
DimGame = DimGame.sort_values('GameName').reset_index(drop=True) 

In [26]:
# Surrogate Key 
DimGame['GameId'] = range(1,len(DimGame)+1) # Warehouse foreign key.

In [27]:
DimGame

,GameName,GameId
0,007 quantum of solace,1
1,007 racing,2
2,007 the world is not enough,3
3,007 tomorrow never dies,4
4,1 vs 100,5
...,...,...
12074,zumba fitness rush,12075
12075,zumba fitness world party,12076
12076,zwei,12077
12077,zyuden sentai kyoryuger game de gaburincho,12078


# MERGING DATASETS (STAGING TABLES)

In [28]:
# Merge engagement dataset to DimGame

engagement_aggregated = engagement_aggregated.merge(
    DimGame,
    left_on='TitleNorm',
    right_on='GameName',
    how='left'
)

In [29]:
engagement_aggregated.head(2)

,TitleNorm,Title,ReleaseDate,Rating,TimesListed,NumberOfReviews,Plays,Playing,Backlogs,Wishlist,GameName,GameId
0,100 orange juice,100% Orange Juice,2009-08-15,3.4,112.0,112.0,1800.0,51.0,292.0,89.0,100 orange juice,10
1,13 sentinels aegis rim,13 Sentinels: Aegis Rim,2019-11-28,4.4,1200.0,1200.0,3700.0,466.0,3300.0,2500.0,13 sentinels aegis rim,23


In [30]:
# Merge Sales dataset to DimGame
df_CleanSalesData = df_CleanSalesData.merge(
    DimGame,
    left_on='NameNorm',
    right_on='GameName',
    how='left'
)

In [31]:
df_CleanSalesData.head(2)

,Rank,Name,Platform,Year,Genre,Publisher,NaSales,EuSales,JpSales,OtherSales,GlobalSales,NameNorm,GameName,GameId
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports,wii sports,11556
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros,super mario bros,9815


In [41]:
# Standardize Publisher

df_CleanSalesData['Publisher'] = (
    df_CleanSalesData['Publisher']
        .str.strip()
        .str.replace('"','', regex=False)
)


# SQL CONNECTION 

In [32]:
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 18 for SQL Server']


In [ ]:
params = urllib.parse.quote_plus(
    "DRIVER=ODBC Driver 18 for SQL Server;"
    "SERVER=ANIRUDH\\SQLEXPRESS;"
    "DATABASE=VideoGame;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

# TEST CONNECTION

In [34]:
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name")
    )
    print(result.fetchone())

('Anirudh\\SQLEXPRESS', 'VideoGameAnalysis')


# LOAD DATA TO SQL

In [ ]:
# engagement_aggregated.to_sql(
#     'df_CleanEngagementData',
#     engine,
#     if_exists='replace',
#     index='False',
#     chunksize=1000
# )

# df_CleanSalesData.to_sql(
#     'df_CleanSalesData',
#     engine,
#     if_exists='replace',
#     index='False',
#     chunksize=1000
# )

# Issue faced - the datatype / autocreated schema was not right. Will have to get data in CSV for further processing

-17

# LOAD DATA TO CSV

In [35]:
# Engagement staging (only needed columns)
StgCleanEngagement = engagement_aggregated[
    [
        'GameId',
        'GameName',
        'ReleaseDate',
        'Rating',
        'TimesListed',
        'NumberOfReviews',
        'Plays',
        'Playing',
        'Backlogs',
        'Wishlist'
    ]
]

StgCleanEngagement.to_csv('StgCleanEngagement.csv', index=False)

In [37]:
StgCleanEngagement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1098 entries, 0 to 1097
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   GameId           1098 non-null   int64  
 1   GameName         1098 non-null   object 
 2   ReleaseDate      1095 non-null   object 
 3   Rating           1085 non-null   float64
 4   TimesListed      1098 non-null   float64
 5   NumberOfReviews  1098 non-null   float64
 6   Plays            1098 non-null   float64
 7   Playing          1098 non-null   float64
 8   Backlogs         1098 non-null   float64
 9   Wishlist         1098 non-null   float64
dtypes: float64(7), int64(1), object(2)
memory usage: 85.9+ KB


In [39]:
len(StgCleanEngagement)

1098

In [42]:
# Sales staging (only needed columns)
StageCleanSales = df_CleanSalesData[
    [
        'GameId',
        'GameName',
        'Platform',
        'Year',
        'Publisher',
        'NaSales',
        'EuSales',
        'JpSales',
        'OtherSales',
        'GlobalSales'
    ]
]

StageCleanSales.to_csv('StageCleanSales.csv', index=False)

In [43]:
StageCleanSales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GameId       16598 non-null  int64  
 1   GameName     16598 non-null  object 
 2   Platform     16598 non-null  object 
 3   Year         16598 non-null  int64  
 4   Publisher    16598 non-null  object 
 5   NaSales      16598 non-null  float64
 6   EuSales      16598 non-null  float64
 7   JpSales      16598 non-null  float64
 8   OtherSales   16598 non-null  float64
 9   GlobalSales  16598 non-null  float64
dtypes: float64(5), int64(2), object(3)
memory usage: 1.3+ MB


In [44]:
len(StageCleanSales)

16598